##Cell 1 — Install + check GPU

In [ ]:
from google.colab import files
files.upload()  # Upload raft_clean.jsonl

!pip -q install unsloth trl datasets accelerate

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

# Optional: avoids some audio dependency conflicts
!pip -q uninstall -y torchaudio


##Cell 2 — Load + format dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="raft_clean.jsonl", split="train")
print("rows:", len(dataset))
print("cols:", dataset.column_names)
print("sample:", dataset[0])

def format_example(ex):
    # Simple “Alpaca-style” template
    return {
        "text": (
            "### Instruction:\n" + ex["instruction"].strip() + "\n\n"
            "### Input:\n" + ex["input"].strip() + "\n\n"
            "### Response:\n" + ex["output"].strip()
        )
    }

dataset = dataset.map(format_example, remove_columns=dataset.column_names)
print(dataset[0]["text"][:500])


##Cell 3 — Load model + LoRA

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True  # saves VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0.0,
)


##Cell 4 — Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForLanguageModeling

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,  # effective batch = 16
        warmup_steps=10,
        max_steps=300,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        output_dir="outputs",
        save_steps=100,
    ),
)

trainer.train()


##Cell 5 — Quick inference test (sanity check)

In [5]:
FastLanguageModel.for_inference(model)

prompt = """### Instruction:
Answer using only the provided context.
### Input:
Context (oracle):
Hypertension guideline: start ACE inhibitor at X dose...
Distractors:
...
### Response:
"""

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=120, temperature=0.2)
print(tokenizer.decode(out[0], skip_special_tokens=True))


### Instruction:
Answer using only the provided context.
### Input:
Context (oracle):
Hypertension guideline: start ACE inhibitor at X dose...
Distractors:
...
### Response:
The ORACLE passage states that the HTWOS study, which examined the long-term effects of rosiglitazone on T2DM, reported a reduction in microvascular complications incidence of 24% over a 15-year period. The study used a similar dose to metformin, which was a reduced form of the drug to prevent money loss.

Final answer: A reduced form of the drug, which was a reduced form of the drug to prevent money loss, a 24% reduction in microvascular complications incidence.


##Cell 6 — Save adapter (LoRA)

In [ ]:
model.save_pretrained("raft_lora_adapter")
tokenizer.save_pretrained("raft_lora_adapter")

!zip -r raft_lora_adapter.zip raft_lora_adapter
!ls -lh raft_lora_adapter.zip


##Cell 7 — OPTIONAL: merge + convert to GGUF + quantize

In [ ]:
# Merge LoRA into a full 16-bit model (optional, needed for GGUF conversion)
model.save_pretrained_merged("merged_model_16bit", tokenizer, save_method="merged_16bit")

!rm -rf /content/llama.cpp
!git clone https://github.com/ggerganov/llama.cpp

!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged_model_16bit \
  --outfile /content/qwen2.5-0.5b-raft-f16.gguf --outtype f16

!ls -lh /content/qwen2.5-0.5b-raft-f16.gguf

from google.colab import files
files.download("/content/qwen2.5-0.5b-raft-f16.gguf")

# Build quantizer + quantize to Q8_0
!rm -rf /content/llama.cpp/build
!cmake -S /content/llama.cpp -B /content/llama.cpp/build \
  -DCMAKE_BUILD_TYPE=Release \
  -DLLAMA_BUILD_TESTS=OFF \
  -DLLAMA_BUILD_EXAMPLES=OFF

!cmake --build /content/llama.cpp/build --target llama-quantize -j 1

!/content/llama.cpp/build/bin/llama-quantize \
  /content/qwen2.5-0.5b-raft-f16.gguf \
  /content/qwen2.5-0.5b-raft-q8_0.gguf \
  q8_0

!ls -lh /content/qwen2.5-0.5b-raft-q8_0.gguf
files.download("/content/qwen2.5-0.5b-raft-q8_0.gguf")
